In [2]:
# -*- coding: utf-8 -*-
import os, re, json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

OUT_DIR = "./shap_cache_and_plots_24labels"
CACHE_DIR = os.path.join(OUT_DIR, "cache")
PLOT_DIR  = os.path.join(OUT_DIR, "plots")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DPI = 600
TOPK_GROUP = 20   # Morgan/KG 各自Top20用于聚合
TOPK_FINAL = 24   # 热图/散点最终Top24特征

# 24个要训练的标签
TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

# 138标签列名：只用于“把它们从X里排除”，避免被当成特征列
LABELS_138 = [
    "alcoholic","aldehydic","alliaceous","almond","amber","animal","anisic","apple","apricot","aromatic",
    "balsamic","banana","beefy","bergamot","berry","bitter","black currant","brandy","burnt","buttery",
    "cabbage","camphoreous","caramellic","cedar","celery","chamomile","cheesy","cherry","chocolate","cinnamon",
    "citrus","clean","clove","cocoa","coconut","coffee","cognac","cooked","cooling","cortex","coumarinic",
    "creamy","cucumber","dairy","dry","earthy","ethereal","fatty","fermented","fishy","floral","fresh",
    "fruit skin","fruity","garlic","gassy","geranium","grape","grapefruit","grassy","green","hawthorn","hay",
    "hazelnut","herbal","honey","hyacinth","jasmin","juicy","ketonic","lactonic","lavender","leafy","leathery",
    "lemon","lily","malty","meaty","medicinal","melon","metallic","milky","mint","muguet","mushroom","musk",
    "musty","natural","nutty","odorless","oily","onion","orange","orangeflower","orris","ozone","peach","pear",
    "phenolic","pine","pineapple","plum","popcorn","potato","powdery","pungent","radish","raspberry","ripe",
    "roasted","rose","rummy","sandalwood","savory","sharp","smoky","soapy","solvent","sour","spicy","strawberry",
    "sulfurous","sweaty","sweet","tea","terpenic","tobacco","tomato","tropical","vanilla","vegetable","vetiver",
    "violet","warm","waxy","weedy","winey","woody"
]

# 最优超参（你给的）
BEST_PARAMS = {
    "n_estimators": 405,
    "max_depth": 3,
    "learning_rate": 0.07906600976507383,
    "subsample": 0.7024318394107314,
    "colsample_bytree": 0.6310703120837248,
    "min_child_weight": 0.5211876757007761,
    "reg_lambda": 0.00324380408262788,
    "reg_alpha": 0.00376863962200334,
    "gamma": 0.8594807940282623,
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",   # 有GPU就改 gpu_hist
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)

# 缓存shap用float16压缩，体积小很多
SHAP_SAVE_DTYPE = np.float16


# =========================
# 1) X / y 构建（不靠0/1推断label，避免丢特征）
# =========================
def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_numeric_or_convertible(series: pd.Series) -> bool:
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True
    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False

def build_X_y(df: pd.DataFrame):
    smiles_col = find_smiles_col(df)

    # y24
    miss24 = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if miss24:
        raise ValueError(f"缺少24个目标标签列：{miss24}")
    y24 = df[TARGET_LABELS_24].fillna(0).astype(int).values

    # X：排除 138标签列 + smiles
    exclude = set([c for c in LABELS_138 if c in df.columns])
    if smiles_col is not None:
        exclude.add(smiles_col)

    feat_cols = [c for c in df.columns if c not in exclude]

    # 去掉无法转数值的列
    bad = []
    feat_cols2 = []
    for c in feat_cols:
        if is_numeric_or_convertible(df[c]):
            feat_cols2.append(c)
        else:
            bad.append(c)
    if bad:
        print(f"[WARN] Dropped non-numeric X columns ({len(bad)}):", bad[:10], "..." if len(bad)>10 else "")

    X_df = df[feat_cols2].copy()
    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X = X_df.fillna(0).astype(np.float32).values
    return X, y24, feat_cols2, smiles_col


# =========================
# 2) Morgan/KG 列检测（尽量鲁棒：名字优先，失败再看取值特征）
# =========================
def _is_binary01(arr: np.ndarray) -> bool:
    u = np.unique(arr)
    return u.size > 0 and set(u.tolist()).issubset({0.0, 1.0})

def detect_morgan_kg_cols(df: pd.DataFrame, feat_cols: list):
    # --- 2.1 名字规则（优先） ---
    def is_morgan_name(col):
        s = str(col).strip().lower()
        if s.isdigit():
            return True
        if re.match(r"^(morgan|ecfp|mfp|fp|bit)[\s_\-]?\d+$", s):
            return True
        if s.startswith(("morgan_","ecfp_","fp_","bit_","mfp_")):
            return True
        return False

    def is_kg_name(col):
        s = str(col).strip().lower()
        if s.startswith("kg_emb_"):
            return True
        if "structkg" in s and any(k in s for k in ["emb", "embed", "dim"]):
            return True
        if ("kg" in s) and any(k in s for k in ["emb", "embed", "embedding"]):
            return True
        return False

    m_cols = [c for c in feat_cols if is_morgan_name(c)]
    k_cols = [c for c in feat_cols if is_kg_name(c)]

    # --- 2.2 若名字抓不到Morgan（重命名很常见），用“海量0/1列”兜底 ---
    # Morgan通常 >=1024 列且全0/1
    if len(m_cols) < 200:
        bin_cols = []
        for c in feat_cols:
            s = df[c]
            if not np.issubdtype(s.dtype, np.number) and s.dtype != bool:
                continue
            arr = pd.Series(s).fillna(0).astype(float).values
            if _is_binary01(arr):
                bin_cols.append(c)
        if len(bin_cols) >= 500:
            m_cols = bin_cols  # 认为这些就是Morgan bits（Rule/FG通常没这么多列）
    # --- 2.3 KG兜底：连续值embedding列数量通常 16~512，且非0/1 ---
    if len(k_cols) < 8:
        cont_cols = []
        for c in feat_cols:
            s = df[c]
            if not np.issubdtype(s.dtype, np.number) and s.dtype != bool:
                continue
            arr = pd.Series(s).fillna(0).astype(float).values
            if _is_binary01(arr):
                continue
            # 连续值且不全常数
            if np.std(arr) > 1e-12 and np.unique(arr).size > 20:
                cont_cols.append(c)
        # 如果连续值列数量适中，则把它们当embedding候选
        if 16 <= len(cont_cols) <= 512:
            k_cols = cont_cols

    return m_cols, k_cols


# =========================
# 3) 训练 + SHAP + 缓存（全样本）
# =========================
def train_one_label(X, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)
    clf = xgb.XGBClassifier(**params)
    clf.fit(X, y_bin)
    return clf

def signed_meanabs_from_shap(sv: np.ndarray):
    meanabs = np.mean(np.abs(sv), axis=0)
    sgn = np.sign(np.sum(sv, axis=0))
    sgn[sgn == 0] = 1.0
    return meanabs * sgn

def save_beeswarm(shap_vals, X_vals, feature_names, out_png, max_display=24):
    plt.figure(figsize=(9.0, 11.0))
    shap.summary_plot(
        shap_vals, X_vals,
        feature_names=feature_names,
        plot_type="dot",
        max_display=max_display,
        show=False
    )
    plt.gcf().subplots_adjust(left=0.38, right=0.98, top=0.98, bottom=0.08)
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()

def plot_heatmap_signed(matrix, row_names, col_names, out_png):
    plt.figure(figsize=(max(10, 0.55*len(col_names)), max(6, 0.40*len(row_names))))
    im = plt.imshow(matrix, aspect="auto", cmap="bwr")
    plt.colorbar(im, label="Signed mean(|SHAP|)  (sign = sign(sum SHAP))")
    plt.xticks(np.arange(len(col_names)), col_names, rotation=90, fontsize=9)
    plt.yticks(np.arange(len(row_names)), row_names, fontsize=10)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(j, i, f"{matrix[i, j]:+.2f}", ha="center", va="center", fontsize=6)
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()


# =========================
# 4) 主流程（一次跑完：训练→shap→选top20聚合→缓存→出图）
# =========================
def main():
    print("[INFO] Reading:", FEATURE_FILE)
    df = pd.read_excel(FEATURE_FILE)

    X, y24, feat_cols, smiles_col = build_X_y(df)
    print(f"[INFO] X={X.shape}, y24={y24.shape}, features={len(feat_cols)}")
    if smiles_col:
        print("[INFO] smiles_col =", smiles_col)

    # 保存X与特征名（以后画图不用重训/重读）
    np.savez_compressed(os.path.join(CACHE_DIR, "X_full.npz"), X=X.astype(np.float16))
    with open(os.path.join(CACHE_DIR, "feature_names.json"), "w", encoding="utf-8") as f:
        json.dump(feat_cols, f, ensure_ascii=False, indent=2)

    # 识别 Morgan/KG 列
    m_cols, k_cols = detect_morgan_kg_cols(df, feat_cols)
    print(f"[INFO] detected Morgan cols = {len(m_cols)}")
    print(f"[INFO] detected KG cols     = {len(k_cols)}")

    # 索引
    col_to_i = {c:i for i,c in enumerate(feat_cols)}
    m_idx_all = [col_to_i[c] for c in m_cols if c in col_to_i]
    k_idx_all = [col_to_i[c] for c in k_cols if c in col_to_i]

    meta = {
        "feature_file": FEATURE_FILE,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "target_labels_24": TARGET_LABELS_24,
        "smiles_col": smiles_col,
        "morgan_cols_detected": m_cols[:20],
        "kg_cols_detected": k_cols[:20],
        "morgan_count": int(len(m_idx_all)),
        "kg_count": int(len(k_idx_all)),
        "best_params": BEST_PARAMS,
        "base_xgb_params_single": BASE_XGB_PARAMS_SINGLE,
        "topk_group": TOPK_GROUP,
    }
    with open(os.path.join(CACHE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    # ============ A) 训练24个模型 + 计算并缓存全量SHAP ============
    meanabs_24xF = np.zeros((len(TARGET_LABELS_24), X.shape[1]), dtype=np.float32)

    for li, lab in enumerate(TARGET_LABELS_24):
        print(f"\n[TRAIN+SHAP] {lab} ({li+1}/{len(TARGET_LABELS_24)})")
        y_bin = y24[:, li]

        model = train_one_label(X, y_bin)
        model_path = os.path.join(CACHE_DIR, f"xgb__{lab}.json")
        model.get_booster().save_model(model_path)

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)  # (n_samples, n_features)

        meanabs = np.mean(np.abs(sv), axis=0).astype(np.float32)
        meanabs_24xF[li, :] = meanabs

        # 缓存全量SHAP（float16压缩）
        np.savez_compressed(
            os.path.join(CACHE_DIR, f"shap_full__{lab}.npz"),
            shap_values=sv.astype(SHAP_SAVE_DTYPE),
        )
        np.save(os.path.join(CACHE_DIR, f"meanabs_full__{lab}.npy"), meanabs)

    np.savez_compressed(os.path.join(CACHE_DIR, "meanabs_24xF.npz"),
                        meanabs_24xF=meanabs_24xF,
                        labels=np.array(TARGET_LABELS_24))
    print("\n[SAVED] All models + full SHAP cache done.")

    # ============ B) 基于训练得到的SHAP，选择Morgan/KG各Top20 ============
    global_meanabs = meanabs_24xF.mean(axis=0)  # (F,)

    m_top = sorted(m_idx_all, key=lambda i: global_meanabs[i], reverse=True)[:TOPK_GROUP] if len(m_idx_all) else []
    k_top = sorted(k_idx_all, key=lambda i: global_meanabs[i], reverse=True)[:TOPK_GROUP] if len(k_idx_all) else []

    pd.DataFrame({"Morgan_top20": [feat_cols[i] for i in m_top]}).to_csv(
        os.path.join(CACHE_DIR, "Morgan_top20_by_SHAP.csv"), index=False, encoding="utf-8-sig"
    )
    pd.DataFrame({"KG_top20": [feat_cols[i] for i in k_top]}).to_csv(
        os.path.join(CACHE_DIR, "KG_top20_by_SHAP.csv"), index=False, encoding="utf-8-sig"
    )
    print("[SAVED] Morgan_top20_by_SHAP.csv / KG_top20_by_SHAP.csv")

    # ============ C) 构建“合并后的特征空间”：去掉全部Morgan/KG列，加入2个聚合列 ============
    m_set = set(m_idx_all)
    k_set = set(k_idx_all)
    drop_set = m_set.union(k_set)

    keep_idx = [i for i in range(X.shape[1]) if i not in drop_set]
    keep_names = [feat_cols[i] for i in keep_idx]

    X_keep = X[:, keep_idx]
    X_m = X[:, m_top].mean(axis=1, keepdims=True) if len(m_top) else np.zeros((X.shape[0],1), dtype=np.float32)
    X_k = X[:, k_top].mean(axis=1, keepdims=True) if len(k_top) else np.zeros((X.shape[0],1), dtype=np.float32)

    X_merged = np.hstack([X_keep, X_m, X_k]).astype(np.float32)
    merged_names = keep_names + [f"Morgan(top{TOPK_GROUP} avg)", f"StructKG(top{TOPK_GROUP} avg)"]

    np.savez_compressed(os.path.join(CACHE_DIR, "X_merged.npz"), X=X_merged.astype(np.float16))
    with open(os.path.join(CACHE_DIR, "feature_names_merged.json"), "w", encoding="utf-8") as f:
        json.dump(merged_names, f, ensure_ascii=False, indent=2)

    # ============ D) 不重训：用“全量SHAP + 聚合SHAP”拼出 merged SHAP，然后选Top24并出图 ============
    meanabs_merged = np.zeros((len(TARGET_LABELS_24), X_merged.shape[1]), dtype=np.float32)
    signed_merged  = np.zeros((len(TARGET_LABELS_24), X_merged.shape[1]), dtype=np.float32)

    # 先构建每个label的 merged SHAP（只在内存里用；同时缓存Top24后再另存）
    merged_shap_list = []

    for li, lab in enumerate(TARGET_LABELS_24):
        pack = np.load(os.path.join(CACHE_DIR, f"shap_full__{lab}.npz"))
        sv_full = pack["shap_values"].astype(np.float32)  # (n, F)

        # keep部分shap
        sv_keep = sv_full[:, keep_idx]

        # 聚合shap（按样本对Top20列求平均）
        sv_m = sv_full[:, m_top].mean(axis=1, keepdims=True) if len(m_top) else np.zeros((X.shape[0],1), dtype=np.float32)
        sv_k = sv_full[:, k_top].mean(axis=1, keepdims=True) if len(k_top) else np.zeros((X.shape[0],1), dtype=np.float32)

        sv_merge = np.hstack([sv_keep, sv_m, sv_k]).astype(np.float32)  # (n, F_merge)
        merged_shap_list.append(sv_merge)

        meanabs_merged[li, :] = np.mean(np.abs(sv_merge), axis=0).astype(np.float32)
        signed_merged[li, :]  = signed_meanabs_from_shap(sv_merge).astype(np.float32)

    # 选Top24（按24标签 mean(|SHAP|) 平均后排序）
    global_meanabs_merge = meanabs_merged.mean(axis=0)
    top_idx = np.argsort(global_meanabs_merge)[::-1][:TOPK_FINAL]
    top_names = [merged_names[i] for i in top_idx]

    # 保存Top24列表
    pd.DataFrame({
        "rank": np.arange(1, TOPK_FINAL+1),
        "feature": top_names,
        "global_mean_abs_shap": global_meanabs_merge[top_idx]
    }).to_csv(os.path.join(CACHE_DIR, f"Top{TOPK_FINAL}_merged_features.csv"),
              index=False, encoding="utf-8-sig")

    # 热图（24 labels × Top24 features）
    heat = signed_merged[:, top_idx]  # (24, 24)
    heat_png = os.path.join(PLOT_DIR, f"Heatmap_24labels_Top{TOPK_FINAL}_merged.png")
    plot_heatmap_signed(heat, TARGET_LABELS_24, top_names, heat_png)

    pd.DataFrame(heat, index=TARGET_LABELS_24, columns=top_names).to_csv(
        os.path.join(PLOT_DIR, f"Heatmap_24labels_Top{TOPK_FINAL}_merged.csv"),
        encoding="utf-8-sig"
    )

    # ============ E) 默认散点图（beeswarm）：每个标签一张（用 merged 空间的 Top24） ============
    X_top = X_merged[:, top_idx]
    for li, lab in enumerate(TARGET_LABELS_24):
        sv_merge = merged_shap_list[li]
        sv_top = sv_merge[:, top_idx]
        out_png = os.path.join(PLOT_DIR, f"Beeswarm__{lab}__Top{TOPK_FINAL}_merged.png")
        save_beeswarm(sv_top, X_top, top_names, out_png, max_display=TOPK_FINAL)

        # 顺便把每个标签 Top24 的 shap 缓存下来（以后画图只load这个，超快）
        np.savez_compressed(
            os.path.join(CACHE_DIR, f"shap_merged_top{TOPK_FINAL}__{lab}.npz"),
            shap_values=sv_top.astype(SHAP_SAVE_DTYPE)
        )

    print("\n[DONE] Everything finished.")
    print("Outputs:")
    print(" - Cache:", CACHE_DIR)
    print(" - Plots:", PLOT_DIR)


if __name__ == "__main__":
    main()

[INFO] Reading: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X=(4952, 2627), y24=(4952, 24), features=2627
[INFO] smiles_col = Canonical_SMILES
[INFO] detected Morgan cols = 2048
[INFO] detected KG cols     = 200

[TRAIN+SHAP] alcoholic (1/24)

[TRAIN+SHAP] aldehydic (2/24)

[TRAIN+SHAP] almond (3/24)

[TRAIN+SHAP] aromatic (4/24)

[TRAIN+SHAP] burnt (5/24)

[TRAIN+SHAP] cabbage (6/24)

[TRAIN+SHAP] cheesy (7/24)

[TRAIN+SHAP] cherry (8/24)

[TRAIN+SHAP] chocolate (9/24)

[TRAIN+SHAP] ethereal (10/24)

[TRAIN+SHAP] fishy (11/24)

[TRAIN+SHAP] fruity (12/24)

[TRAIN+SHAP] garlic (13/24)

[TRAIN+SHAP] grassy (14/24)

[TRAIN+SHAP] green (15/24)

[TRAIN+SHAP] ketonic (16/24)

[TRAIN+SHAP] musty (17/24)

[TRAIN+SHAP] pungent (18/24)

[TRAIN+SHAP] sharp (19/24)

[TRAIN+SHAP] solvent (20/24)

[TRAIN+SHAP] sour (21/24)

[TRAIN+SHAP] sulfurous (22/24)

[TRAIN+SHAP] sweaty (23/24)

[TRAIN+SHAP] sweet (24/24)

[SAVED] All models + full SHAP cache done.
[SAVED] Morgan_top20_by_SHAP.csv 